# Week 11: AI Recommendation Engine

Build logic for Buy/Sell/Hold decisions: Combine predicted return 

Design decision rules: Example: Buy if pred_return > 2%

First version of AI investor bot working.

Outputs recommendation with justification.


# Week 12: Final Report, UI & Presentation
Tasks:
Create final project report (PDF or notebook):


Methodology, analysis, ML models, sentiment impact


Prepare slides for final presentation.


Finalize Streamlit/Gradio app (optional).


Record demo (optional) and publish repo.


Milestones:
Complete GitHub repo with code + documentation.


Report, presentation, and optional web app submitted.


In [1]:
# 1) Task 1  — Build 'pred_combined' (pred_* avg → y_pred → lagged log return)
# 2) Task 2  — Apply decision rules (BUY if pred > +2%, SELL if pred < -2%, else HOLD)
# 3) Milestone 1 — First working AI investor bot (end-to-end)
# 4) Milestone 2 — Outputs recommendation with justification (saved snapshot)

# Outputs:
# - week11/task1_combined_pred_latest.csv
# - week11/task2_recommendations.csv / .md
# - week11/ai_bot_m1_recommendations.csv / .md
# - week11/milestone2_recommendations.csv / .md

import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

REPORT_DIR = Path("week10")
OUT_DIR    = Path("week11")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# thresholds (log-return units; 0.002 ≈ +0.2%)
POS_THRESHOLD = 0.002
NEG_THRESHOLD = -0.002
ALLOW_SHORT   = False  # long-only; SELL => avoid/close long (no short)


def load_eval():
    """Load evaluation table (Parquet with CSV fallback), ensure Date dtype, sort."""
    pq, csv = REPORT_DIR / "eval_table.parquet", REPORT_DIR / "eval_table.csv"
    if pq.exists():
        try:
            df = pd.read_parquet(pq).copy()
        except Exception:
            if csv.exists():
                df = pd.read_csv(csv, parse_dates=["Date"]).copy()
            else:
                raise
    elif csv.exists():
        df = pd.read_csv(csv, parse_dates=["Date"]).copy()
    else:
        raise FileNotFoundError("Neither eval_table.parquet nor eval_table.csv found in week10/")
    if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df.sort_values(["Ticker","Date"]).reset_index(drop=True)

def compute_pred_combined(df: pd.DataFrame) -> pd.Series:
    """
    Priority:
      1) mean of pred_* columns (excluding y_pred) if any
      2) else y_pred if present
      3) else naive = yesterday's log return from Close
    """
    pred_cols = [c for c in df.columns
                 if c.lower().startswith("pred_") and c != "y_pred"
                 and pd.api.types.is_numeric_dtype(df[c])]
    if pred_cols:
        return df[pred_cols].mean(axis=1)
    if "y_pred" in df.columns and pd.api.types.is_numeric_dtype(df["y_pred"]):
        return df["y_pred"]
    if "Close" in df.columns:
        logret = np.log(df["Close"] / df["Close"].shift(1))
        return logret.shift(1)
    raise ValueError("No prediction columns found and 'Close' missing; cannot build pred_combined.")

def decide_action(pred: float, pos_thr: float, neg_thr: float, allow_short: bool):
    """
    Stateless decision: returns (action, justification).
    action ∈ {"BUY","SELL","HOLD"}; with shorting OFF, SELL means avoid/close long.
    """
    if pred > pos_thr:
        return (
            "BUY",
            f"Predicted next-day log return {pred:.2%} > {pos_thr:.2%} threshold."
        )
    if pred < neg_thr:
        if allow_short:
            return (
                "SELL",
                f"Predicted next-day log return {pred:.2%} < {neg_thr:.2%}; open short."
            )
        else:
            return (
                "SELL",
                f"Predicted next-day log return {pred:.2%} < {neg_thr:.2%}; avoid/close long (no short)."
            )
    return (
        "HOLD",
        f"Predicted return within dead-zone [{neg_thr:.2%}, {pos_thr:.2%}]."
    )

# =============================================================================
# Combine predicted return
# =============================================================================
eval_df = load_eval()
eval_df["pred_combined"] = compute_pred_combined(eval_df)

task1_latest = (
    eval_df.groupby("Ticker", as_index=False)
           .tail(1)[["Ticker","Date","pred_combined"]]
           .sort_values("Ticker")
           .reset_index(drop=True)
)

display(Markdown("## Task 1 — Latest combined predicted next-day log return"))
display(task1_latest)

task1_csv = OUT_DIR / "task1_combined_pred_latest.csv"
task1_latest.to_csv(task1_csv, index=False)
print("Saved:", task1_csv)

# =============================================================================
# Design decision rules (±2% example)
# =============================================================================
task2_rows = []
for _, r in task1_latest.iterrows():
    pred = float(r["pred_combined"])
    action, reason = decide_action(pred, POS_THRESHOLD, NEG_THRESHOLD, ALLOW_SHORT)
    task2_rows.append({
        "Ticker": r["Ticker"],
        "Date":   pd.to_datetime(r["Date"]).date(),
        "pred_combined": pred,
        "Decision": action,
        "Justification": reason
    })
task2_recs = pd.DataFrame(task2_rows).sort_values("Ticker")

display(Markdown(f"## Task 2 — Recommendations (long-only)  \n"
                 f"Rule: BUY if pred > +{POS_THRESHOLD:.0%}, SELL if pred < {NEG_THRESHOLD:.0%}, else HOLD."))
display(task2_recs)

task2_csv = OUT_DIR / "task2_recommendations.csv"
task2_md  = OUT_DIR / "task2_recommendations.md"
task2_recs.to_csv(task2_csv, index=False)
md_lines = [
    "# Task 2 — Recommendations (long-only)\n\n",
    f"Rule: **BUY** if pred_combined > +{POS_THRESHOLD:.0%}, **SELL** if pred_combined < {NEG_THRESHOLD:.0%}, else **HOLD**.\n\n",
    "| Ticker | Date | pred_combined | Decision | Justification |\n",
    "|---|---|---:|---|---|\n",
]
for _, rr in task2_recs.iterrows():
    md_lines.append(f"| {rr.Ticker} | {rr.Date} | {rr.pred_combined:.5f} | {rr.Decision} | {rr.Justification} |\n")
task2_md.write_text("".join(md_lines), encoding="utf-8")
print("Saved:", task2_csv, "and", task2_md)

# =============================================================================
# First working AI investor bot (end-to-end)
# =============================================================================
m1_rows = []
for _, r in task1_latest.iterrows():
    pred = float(r["pred_combined"])
    action, reason = decide_action(pred, POS_THRESHOLD, NEG_THRESHOLD, ALLOW_SHORT)
    m1_rows.append({
        "Ticker": r["Ticker"],
        "Date":   pd.to_datetime(r["Date"]).date(),
        "pred_combined": pred,
        "Decision": action,
        "Justification": reason
    })
m1_recs = pd.DataFrame(m1_rows).sort_values("Ticker")

display(Markdown("## Milestone 1 — AI Investor Bot (2% / -2% rule, long-only)"))
display(m1_recs)

m1_csv = OUT_DIR / "ai_bot_m1_recommendations.csv"
m1_md  = OUT_DIR / "ai_bot_m1_recommendations.md"
m1_recs.to_csv(m1_csv, index=False)

md = [
    "# Milestone 1 — AI Investor Bot (2% / -2% rule, long-only)\n\n",
    f"Rule: **BUY** if pred_combined > +{POS_THRESHOLD:.0%}, **SELL** if pred_combined < {NEG_THRESHOLD:.0%}, else **HOLD**.\n\n",
    "| Ticker | Date | pred_combined | Decision | Justification |\n",
    "|---|---|---:|---|---|\n",
]
for _, rr in m1_recs.iterrows():
    md.append(f"| {rr.Ticker} | {rr.Date} | {rr.pred_combined:.5f} | {rr.Decision} | {rr.Justification} |\n")
Path(m1_md).write_text("".join(md), encoding="utf-8")
print("Saved:", m1_csv, "and", m1_md)

# =============================================================================
# Outputs recommendation with justification
# =============================================================================
m2_recs = m1_recs.copy()

display(Markdown("## Milestone 2 — Recommendations with justification (±2% rule)"))
display(m2_recs)

m2_csv = OUT_DIR / "milestone2_recommendations.csv"
m2_md  = OUT_DIR / "milestone2_recommendations.md"
m2_recs.to_csv(m2_csv, index=False)

md2 = [
    "# Milestone 2 — Recommendations with justification (±2% rule)\n\n",
    f"Rule: **BUY** if pred_combined > +{POS_THRESHOLD:.0%}, **SELL** if pred_combined < {NEG_THRESHOLD:.0%}, else **HOLD**.\n\n",
    "| Ticker | Date | pred_combined | Decision | Justification |\n",
    "|---|---|---:|---|---|\n",
]
for _, rr in m2_recs.iterrows():
    md2.append(f"| {rr.Ticker} | {rr.Date} | {rr.pred_combined:.5f} | {rr.Decision} | {rr.Justification} |\n")
Path(m2_md).write_text("".join(md2), encoding="utf-8")
print("Saved:", m2_csv, "and", m2_md)

## Task 1 — Latest combined predicted next-day log return

,Ticker,Date,pred_combined
0,AAPL,2025-07-09,0.000286
1,AMZN,2025-07-09,-0.018563
2,CRM,2025-07-09,0.014169
3,GOOGL,2025-07-09,-0.013840
4,IBM,2025-07-09,-0.007034
5,META,2025-07-09,0.003224
6,MSFT,2025-07-09,-0.002213
7,NVDA,2025-07-08,-0.006927
8,ORCL,2025-07-08,-0.021552
9,TSLA,2025-07-09,0.013080


Saved: week11/task1_combined_pred_latest.csv


## Task 2 — Recommendations (long-only)  
Rule: BUY if pred > +0%, SELL if pred < -0%, else HOLD.

,Ticker,Date,pred_combined,Decision,Justification
0,AAPL,2025-07-09,0.000286,HOLD,"Predicted return within dead-zone [-0.20%, 0.2..."
1,AMZN,2025-07-09,-0.018563,SELL,Predicted next-day log return -1.86% < -0.20%;...
2,CRM,2025-07-09,0.014169,BUY,Predicted next-day log return 1.42% > 0.20% th...
3,GOOGL,2025-07-09,-0.013840,SELL,Predicted next-day log return -1.38% < -0.20%;...
4,IBM,2025-07-09,-0.007034,SELL,Predicted next-day log return -0.70% < -0.20%;...
5,META,2025-07-09,0.003224,BUY,Predicted next-day log return 0.32% > 0.20% th...
6,MSFT,2025-07-09,-0.002213,SELL,Predicted next-day log return -0.22% < -0.20%;...
7,NVDA,2025-07-08,-0.006927,SELL,Predicted next-day log return -0.69% < -0.20%;...
8,ORCL,2025-07-08,-0.021552,SELL,Predicted next-day log return -2.16% < -0.20%;...
9,TSLA,2025-07-09,0.013080,BUY,Predicted next-day log return 1.31% > 0.20% th...


...

,Ticker,Date,pred_combined,Decision,Justification
0,AAPL,2025-07-09,0.000286,HOLD,"Predicted return within dead-zone [-0.20%, 0.2..."
1,AMZN,2025-07-09,-0.018563,SELL,Predicted next-day log return -1.86% < -0.20%;...
2,CRM,2025-07-09,0.014169,BUY,Predicted next-day log return 1.42% > 0.20% th...
3,GOOGL,2025-07-09,-0.013840,SELL,Predicted next-day log return -1.38% < -0.20%;...
4,IBM,2025-07-09,-0.007034,SELL,Predicted next-day log return -0.70% < -0.20%;...
5,META,2025-07-09,0.003224,BUY,Predicted next-day log return 0.32% > 0.20% th...
6,MSFT,2025-07-09,-0.002213,SELL,Predicted next-day log return -0.22% < -0.20%;...
7,NVDA,2025-07-08,-0.006927,SELL,Predicted next-day log return -0.69% < -0.20%;...
8,ORCL,2025-07-08,-0.021552,SELL,Predicted next-day log return -2.16% < -0.20%;...
9,TSLA,2025-07-09,0.013080,BUY,Predicted next-day log return 1.31% > 0.20% th...


Saved: week11/ai_bot_m1_recommendations.csv and week11/ai_bot_m1_recommendations.md


## Milestone 2 — Recommendations with justification (±2% rule)

,Ticker,Date,pred_combined,Decision,Justification
0,AAPL,2025-07-09,0.000286,HOLD,"Predicted return within dead-zone [-0.20%, 0.2..."
1,AMZN,2025-07-09,-0.018563,SELL,Predicted next-day log return -1.86% < -0.20%;...
2,CRM,2025-07-09,0.014169,BUY,Predicted next-day log return 1.42% > 0.20% th...
3,GOOGL,2025-07-09,-0.013840,SELL,Predicted next-day log return -1.38% < -0.20%;...
4,IBM,2025-07-09,-0.007034,SELL,Predicted next-day log return -0.70% < -0.20%;...
5,META,2025-07-09,0.003224,BUY,Predicted next-day log return 0.32% > 0.20% th...
6,MSFT,2025-07-09,-0.002213,SELL,Predicted next-day log return -0.22% < -0.20%;...
7,NVDA,2025-07-08,-0.006927,SELL,Predicted next-day log return -0.69% < -0.20%;...
8,ORCL,2025-07-08,-0.021552,SELL,Predicted next-day log return -2.16% < -0.20%;...
9,TSLA,2025-07-09,0.013080,BUY,Predicted next-day log return 1.31% > 0.20% th...


Saved: week11/milestone2_recommendations.csv and week11/milestone2_recommendations.md


In [2]:
pip install gradio pandas numpy

Looking in indexes: https://mirrors.bfsu.edu.cn/pypi/web/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 MB 10.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.7/815.7 kB 19.2 MB/s eta 0:00:00
  Using cached https://mirrors.bfsu.edu.cn/pypi/web/packages/e5/47/d63c60f59a59467fda0f93f46335c9d18526d7071f025cb5b89d5353ea42/fastapi-0.116.1-py3-none-any.whl (95 kB)
  Using cached https://mirrors.bfsu.edu.cn/pypi/web/packages/74/d4/1806897b31c480efc4e97c22506ac46c716084f573aef780bb7fb7a16e8a/ffmpy-0.6.1-py3-none-any.whl (5.5 kB)
  Using cached https://mirrors.bfsu.edu.cn/pypi/web/packages/28/27/3d6dcadc8a3214d8522c1e7f6a19554e33659be44546d44a2f7572ac7d2a/groovy-0.1.2-py3-none-any.whl (14 kB)
  Using cached https://mirrors.bfsu.edu.cn/pypi/web/packages/39/7b/bb06b061991107cd8783f300adff3e7b7f284e330fd82f507f2a1417b11d/huggingface_hub-0.34.4-py3-none-any.whl (561 kB)
  Using cached https://mirrors.bfsu.edu.cn/pypi/web/packages/a6/53/d78dc063216e62fc55f

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.6/199.6 kB 18.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 31.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.2/107.2 kB 3.5 MB/s eta 0:00:00
  Using cached https://mirrors.bfsu.edu.cn/pypi/web/packages/e0/f9/0595336914c5619e5f28a1fb793285925a8cd4b432c9da0a987836c7f822/shellingham-1.5.4-py2.py3-none-any.whl (9.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.7.1
    Uninstalling typing_extensions-4.7.1:
      Successfully uninstalled typing_extensions-4.7.1
  Attempting uninstall: tomlkit
    Found existing i

In [3]:
# gradio
import io
import gradio as gr

REPORT_DIR = Path("week10")

def load_eval():
    pq, csv = REPORT_DIR / "eval_table.parquet", REPORT_DIR / "eval_table.csv"
    if pq.exists():
        try:
            df = pd.read_parquet(pq).copy()
        except Exception:
            if csv.exists():
                df = pd.read_csv(csv, parse_dates=["Date"]).copy()
            else:
                raise
    elif csv.exists():
        df = pd.read_csv(csv, parse_dates=["Date"]).copy()
    else:
        raise FileNotFoundError("Neither eval_table.parquet nor eval_table.csv found in week10/")
    if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    return df.sort_values(["Ticker","Date"]).reset_index(drop=True)

def ensure_truth(df):
    if "target_next_log_return" in df.columns and pd.api.types.is_numeric_dtype(df["target_next_log_return"]):
        return df["target_next_log_return"]
    if "Close" in df.columns:
        logret = np.log(df["Close"] / df["Close"].shift(1))
        return logret.shift(-1)
    raise ValueError("Need target_next_log_return or Close for y_true")

def compute_pred_combined(df):
    pred_cols = [c for c in df.columns if c.lower().startswith("pred_") and c != "y_pred"
                 and pd.api.types.is_numeric_dtype(df[c])]
    if pred_cols: return df[pred_cols].mean(axis=1)
    if "y_pred" in df.columns and pd.api.types.is_numeric_dtype(df["y_pred"]): return df["y_pred"]
    if "Close" in df.columns:
        logret = np.log(df["Close"] / df["Close"].shift(1))
        return logret.shift(1)
    raise ValueError("No prediction columns and no Close; cannot build pred_combined.")

def decide_action(pred, pos_thr, neg_thr, allow_short):
    if pred > pos_thr:
        return ("BUY",  f"Pred {pred:.2%} > +{pos_thr:.2%}.")
    if pred < neg_thr:
        if allow_short: return ("SELL", f"Pred {pred:.2%} < {neg_thr:.2%}; open short.")
        return ("SELL", f"Pred {pred:.2%} < {neg_thr:.0%}; avoid/close long (no short).")
    return ("HOLD", f"Pred within dead-zone [{neg_thr:.0%}, +{pos_thr:.2%}].")

def run_bot(pos_thr_pct, neg_thr_pct, allow_short):
    pos_thr = pos_thr_pct / 100.0
    neg_thr = neg_thr_pct / 100.0

    df = load_eval()
    if "pred_combined" not in df.columns:
        df["pred_combined"] = compute_pred_combined(df)

    latest = df.groupby("Ticker", as_index=False).tail(1).reset_index(drop=True)

    rows = []
    for _, r in latest.iterrows():
        pred = float(r["pred_combined"])
        action, reason = decide_action(pred, pos_thr, neg_thr, allow_short)
        rows.append({
            "Ticker": r["Ticker"],
            "Date":   pd.to_datetime(r["Date"]).date(),
            "pred_combined": pred,
            "Decision": action,
            "Justification": reason
        })
    recs = pd.DataFrame(rows).sort_values("Ticker")

    csv_buf = io.StringIO()
    recs.to_csv(csv_buf, index=False)
    return recs, csv_buf.getvalue()

with gr.Blocks(title="AI Investor Bot") as demo:
    gr.Markdown("# AI Investor Bot — Quick Recommendations")
    with gr.Row():
        pos_thr = gr.Slider(0.0, 3.0, value=0.2, step=0.05, label="BUY threshold (+%)")
        neg_thr = gr.Slider(-3.0, 0.0, value=-0.2, step=0.05, label="SELL threshold (−%)")
        allow_s = gr.Checkbox(label="Allow shorting", value=False)
    run_btn = gr.Button("Run")
    out_tbl = gr.Dataframe(headers=["Ticker","Date","pred_combined","Decision","Justification"], wrap=True)
    out_csv = gr.File(label="Download CSV", visible=False)

    def _on_run(p, n, s):
        df, csv_text = run_bot(p, n, s)
        tmp = Path("latest_orders.csv")
        df.to_csv(tmp, index=False)
        return df, str(tmp)

    run_btn.click(_on_run, inputs=[pos_thr, neg_thr, allow_s], outputs=[out_tbl, out_csv])

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# Update

In [11]:
# AI Investor Bot — Time-travel Portfolio (quality gate + risk-aware sizing)

from __future__ import annotations

import io
from pathlib import Path
import numpy as np
import pandas as pd
import gradio as gr

REPORT_DIR = Path("week10")
OUT_DIR    = Path("week11")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def load_eval() -> pd.DataFrame:
    """Load eval table (Parquet with CSV fallback), ensure Date dtype, sort."""
    pq, csv = REPORT_DIR / "eval_table.parquet", REPORT_DIR / "eval_table.csv"
    if pq.exists():
        try:
            df = pd.read_parquet(pq).copy()
        except Exception:
            if csv.exists():
                df = pd.read_csv(csv, parse_dates=["Date"]).copy()
            else:
                raise
    elif csv.exists():
        df = pd.read_csv(csv, parse_dates=["Date"]).copy()
    else:
        raise FileNotFoundError("Neither eval_table.parquet nor eval_table.csv found in week10/")

    if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    return df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

def ensure_truth(df: pd.DataFrame) -> pd.Series:
    """Return y_true = next-day log return (no leakage)."""
    if "target_next_log_return" in df.columns and pd.api.types.is_numeric_dtype(df["target_next_log_return"]):
        return df["target_next_log_return"]
    if "Close" in df.columns:
        logret = np.log(df["Close"] / df["Close"].shift(1))
        return logret.shift(-1)
    raise ValueError("Need target_next_log_return or Close for y_true.")

def compute_pred_combined(df: pd.DataFrame) -> pd.Series:
    """
    Priority:
      1) mean of pred_* columns (excluding y_pred) if any
      2) else y_pred if present
      3) else naive = yesterday's log return from Close
    """
    pred_cols = [
        c for c in df.columns
        if c.lower().startswith("pred_")
        and c != "y_pred"
        and pd.api.types.is_numeric_dtype(df[c])
    ]
    if pred_cols:
        return df[pred_cols].mean(axis=1)

    if "y_pred" in df.columns and pd.api.types.is_numeric_dtype(df["y_pred"]):
        return df["y_pred"]

    if "Close" in df.columns:
        logret = np.log(df["Close"] / df["Close"].shift(1))
        return logret.shift(1)

    raise ValueError("No prediction columns and no Close; cannot build pred_combined.")

def add_rolling_corr(df: pd.DataFrame, win: int, minp: int) -> pd.DataFrame:
    """Rolling corr(pred_combined, y_true) per ticker (no deprecated groupby.apply)."""
    df = df.copy()
    df["RollCorr"] = np.nan
    for _, idx in df.groupby("Ticker").groups.items():
        g = df.loc[idx].sort_values("Date")
        rc = g["pred_combined"].rolling(win, min_periods=minp).corr(g["y_true"])
        df.loc[g.index, "RollCorr"] = rc.values
    return df

# -----------------------------
# risk utilities
# -----------------------------
def daily_to_annual_vol(sig_daily: float) -> float:
    return float(sig_daily) * np.sqrt(252.0)

def annual_to_daily_vol(sig_annual: float) -> float:
    return float(sig_annual) / np.sqrt(252.0)

def cap_and_renorm(weights, cap=0.25, max_iter=20, tol=1e-12):
    """Cap weights at 'cap' and renormalize the rest (long-only)."""
    w = np.array(weights, dtype=float)
    if w.sum() <= 0:
        return np.zeros_like(w)
    w = w / w.sum()
    for _ in range(max_iter):
        over = w > cap
        if not np.any(over):
            break
        excess = w[over] - cap
        w[over] = cap
        remain = ~over
        rem_sum = w[remain].sum()
        if rem_sum < tol:
            break
        w[remain] = w[remain] + (w[remain] / rem_sum) * excess.sum()
    w = np.clip(w, 0.0, 1.0)
    s = w.sum()
    return w / s if s > tol else w

def build_base_weights(method, tickers, vol_series, preds=None, buy_thr=0.0):
    """
    method ∈ {"equal", "inverse_vol", "signal_vol"}
    - inverse_vol: weight ∝ 1/σ
    - signal_vol : weight ∝ max(pred - thr, 0) / σ
    """
    n = len(tickers)
    if n == 0:
        return np.array([])

    if method == "equal":
        return np.ones(n) / n

    if method in ("inverse_vol", "signal_vol"):
        v = np.array([max(vol_series.get(t, np.nan), np.nan) for t in tickers], dtype=float)
        if np.isnan(v).any() or (v <= 0).any():
            safe = np.where((v > 0) & np.isfinite(v), v, np.nan)
            fallback = np.nanmedian(safe) if np.isfinite(np.nanmedian(safe)) else 0.02
            v = np.where((v > 0) & np.isfinite(v), v, fallback)

        if method == "inverse_vol":
            inv = 1.0 / v
            return inv / inv.sum()

        # signal_vol
        s = np.array([max(float(preds[t]) - buy_thr, 0.0) for t in tickers], dtype=float)
        raw = s / v
        if raw.sum() <= 0:
            return np.ones(n) / n
        return raw / raw.sum()

    raise ValueError("Unknown method: use 'equal', 'inverse_vol', or 'signal_vol'.")

def portfolio_as_of(as_of_date: pd.Timestamp,
                    capital_usd: float,
                    buy_thr_pct: float,
                    corr_min: float,
                    method: str,
                    max_single_w_pct: float,
                    target_ann_vol_pct: float,
                    roll_corr_win: int,
                    roll_cov_win: int,
                    min_periods: int):
    """
    Returns: candidates_df, alloc_df, summary_df
    """
    pos_thr = buy_thr_pct / 100.0
    target_ann_vol = target_ann_vol_pct / 100.0
    max_single_w = max_single_w_pct / 100.0

    # Load & prep
    df = load_eval()
    if "pred_combined" not in df.columns:
        df["pred_combined"] = compute_pred_combined(df)
    if "y_true" not in df.columns:
        df["y_true"] = ensure_truth(df)

    df = df[df["Date"] <= pd.to_datetime(as_of_date)].copy()
    if df.empty:
        raise ValueError("No data up to the selected as-of date. Pick a later date.")
    df = add_rolling_corr(df, win=roll_corr_win, minp=min_periods)

    latest = (
        df.sort_values("Date")
          .groupby("Ticker", as_index=False)
          .tail(1)[["Ticker", "Date", "Close", "pred_combined", "RollCorr"]]
          .sort_values("Ticker")
          .reset_index(drop=True)
    )

    latest["Signal"] = (latest["pred_combined"] > pos_thr).astype(int)
    candidates = latest[(latest["Signal"] == 1) & (latest["RollCorr"] >= corr_min)].copy()
    pivot = df.pivot_table(index="Date", columns="Ticker", values="y_true", aggfunc="first").sort_index()
    win = pivot.tail(roll_cov_win)
    cov_all = win.cov(min_periods=min_periods)
    vol_all = win.std(axis=0)

    if candidates.empty:
        empty_alloc = pd.DataFrame(columns=["Ticker","Close","pred","actual_w","dollars","shares"])
        summary = pd.DataFrame([{
            "Capital ($)": round(capital_usd, 2),
            "Invested ($)": 0.0,
            "Cash ($)": round(capital_usd, 2),
            "Cash weight": 1.0,
            "Exp μ (daily, %)": 0.0,
            "Exp μ (annual, %)": 0.0,
            "Vol (daily, %)": 0.0,
            "Vol (annual, %)": 0.0,
            "Exp Sharpe": np.nan,
            "1-day 95% VaR ($)": 0.0,
            "Method": method,
            "Target Ann Vol": target_ann_vol,
            "Max Single W": max_single_w,
            "BUY thr": pos_thr,
            "Corr gate": corr_min,
        }])
        return (
            candidates[["Ticker","Date","Close","pred_combined","RollCorr"]],
            empty_alloc, summary
        )

    tickers = candidates["Ticker"].tolist()
    cov = cov_all.reindex(index=tickers, columns=tickers).copy()
    vol = vol_all.reindex(index=tickers)

    for t in tickers:
        if not np.isfinite(cov.loc[t, t]):
            var = float(vol.loc[t]**2) if np.isfinite(vol.loc[t]) else 0.0001
            cov.loc[t, t] = var
    cov = cov.fillna(0.0)

    closes = candidates.set_index("Ticker")["Close"].to_dict()
    preds  = candidates.set_index("Ticker")["pred_combined"].to_dict()

    base_w = build_base_weights(method, tickers, vol, preds=preds, buy_thr=pos_thr)
    base_w = cap_and_renorm(base_w, cap=max_single_w)

    # Vol targeting scale
    W = np.array(base_w, dtype=float)
    Sigma = cov.values
    port_vol_daily = float(np.sqrt(max(W @ Sigma @ W, 0.0)))
    target_daily = annual_to_daily_vol(target_ann_vol)
    scale = min(1.0, (target_daily / port_vol_daily)) if port_vol_daily > 0 else 0.0
    final_w = W * scale
    dollars = {t: float(final_w[i] * capital_usd) for i, t in enumerate(tickers)}
    shares  = {t: int(np.floor(dollars[t] / closes[t])) for t in tickers}
    used    = {t: shares[t] * closes[t] for t in tickers}

    invested_usd = float(np.sum(list(used.values())))
    cash_usd     = float(capital_usd - invested_usd)
    actual_w     = np.array([used[t] / capital_usd for t in tickers])
    actual_cash_w = 1.0 - actual_w.sum()

    actual_vol_daily = float(np.sqrt(max(actual_w @ Sigma @ actual_w, 0.0)))
    mu_daily = float(np.sum([preds[t] * (used[t] / capital_usd) for t in tickers]))
    mu_ann   = mu_daily * 252.0
    vol_ann  = daily_to_annual_vol(actual_vol_daily)
    exp_sharpe = (mu_daily / actual_vol_daily) * np.sqrt(252.0) if actual_vol_daily > 0 else np.nan
    var95_1d_usd = 1.65 * actual_vol_daily * capital_usd

    alloc_rows = []
    for t in tickers:
        alloc_rows.append({
            "Ticker": t,
            "Close": closes[t],
            "pred": preds[t],
            "actual_w": used[t] / capital_usd,
            "dollars": used[t],
            "shares": shares[t],
        })
    alloc_df = pd.DataFrame(alloc_rows).sort_values("Ticker").reset_index(drop=True)

    summary = pd.DataFrame([{
        "Capital ($)": round(capital_usd, 2),
        "Invested ($)": round(invested_usd, 2),
        "Cash ($)": round(cash_usd, 2),
        "Cash weight": round(actual_cash_w, 4),
        "Exp μ (daily, %)": round(mu_daily*100, 3),
        "Exp μ (annual, %)": round(mu_ann*100, 2),
        "Vol (daily, %)": round(actual_vol_daily*100, 3),
        "Vol (annual, %)": round(vol_ann*100, 2),
        "Exp Sharpe": round(exp_sharpe, 3) if np.isfinite(exp_sharpe) else np.nan,
        "1-day 95% VaR ($)": round(var95_1d_usd, 2),
        "Method": method,
        "Target Ann Vol": target_ann_vol,
        "Max Single W": max_single_w,
        "BUY thr": pos_thr,
        "Corr gate": corr_min,
    }])

    return (
        candidates[["Ticker","Date","Close","pred_combined","RollCorr"]],
        alloc_df[["Ticker","Close","pred","actual_w","dollars","shares"]],
        summary
    )

def save_slide(as_of: pd.Timestamp, alloc_df: pd.DataFrame, summary_df: pd.DataFrame):
    """Save dated CSV + Markdown for presentations."""
    csv_path = OUT_DIR / f"portfolio_plan_{as_of.date()}.csv"
    md_path  = OUT_DIR / f"portfolio_plan_{as_of.date()}.md"

    alloc_df.to_csv(csv_path, index=False)

    md = []
    md.append(f"# Portfolio Plan — {as_of.date()}\n\n")
    md.append("## Allocation\n\n")
    md.append("| Ticker | Close | pred | weight | dollars | shares |\n|---|---:|---:|---:|---:|---:|\n")
    for r in alloc_df.itertuples(index=False):
        md.append(f"| {r.Ticker} | {r.Close:.2f} | {r.pred:+.3%} | {r.actual_w:.2%} | ${r.dollars:,.2f} | {r.shares} |\n")
    md.append("\n## Summary\n\n")
    for k, v in summary_df.iloc[0].items():
        md.append(f"- **{k}**: {v}\n")
    Path(md_path).write_text("".join(md), encoding="utf-8")

    return str(csv_path), str(md_path)

try:
    _df0 = load_eval()
    DATA_MIN = pd.to_datetime(_df0["Date"].min()).normalize()
    DATA_MAX = pd.to_datetime(_df0["Date"].max()).normalize()
except Exception:
    DATA_MIN = None
    DATA_MAX = None

def clamp_date(d: pd.Timestamp) -> pd.Timestamp:
    if DATA_MIN is not None and d < DATA_MIN:
        return DATA_MIN
    if DATA_MAX is not None and d > DATA_MAX:
        return DATA_MAX
    return d

with gr.Blocks(title="AI Investor Bot — Time-travel Portfolio") as demo:
    gr.Markdown("# AI Investor Bot — Time-travel Portfolio")
    gr.Markdown(
        "Pick an **as-of date** (or use presets), then size a portfolio from the latest predictions "
        "available up to that date with a **quality gate** (rolling corr) and **target vol**."
    )

    with gr.Row():
        asof_txt  = gr.Textbox(label="As-of date (YYYY-MM-DD)", placeholder="e.g., 2024-06-30")
        btn_latest = gr.Button("Latest")
        btn_2y     = gr.Button("−2 years")
        btn_2m     = gr.Button("−2 months")

    with gr.Row():
        with gr.Column(scale=1):
            capital   = gr.Number(label="Capital ($)", value=20000, precision=0)
            pos_thr   = gr.Slider(0.0, 1.0, value=0.20, step=0.05, label="BUY threshold (+%, of next-day log return)")
            corr_gate = gr.Slider(0.00, 0.20, value=0.07, step=0.01, label="Quality gate: Rolling corr ≥")
            method    = gr.Dropdown(choices=["signal_vol","inverse_vol","equal"], value="signal_vol", label="Weight method")
            max_w     = gr.Slider(5, 50, value=25, step=1, label="Max single weight (%)")
            targ_vol  = gr.Slider(2.0, 30.0, value=10.0, step=0.5, label="Target annual volatility (%)")
        with gr.Column(scale=1):
            roll_corr = gr.Slider(20, 120, value=60, step=5, label="Rolling corr window (days)")
            roll_cov  = gr.Slider(20, 120, value=60, step=5, label="Rolling covariance window (days)")
            minp      = gr.Slider(10, 90, value=40, step=5, label="Min periods for stats")
            run_btn   = gr.Button("Run", variant="primary")

            out_csv = gr.File(label="Download allocation CSV")
            out_md  = gr.File(label="Download plan Markdown")

    gr.Markdown("### Candidates (BUY signals after quality gate)")
    out_candidates = gr.Dataframe(wrap=True)
    gr.Markdown("### Allocation (after risk scaling & share rounding)")
    out_alloc = gr.Dataframe(wrap=True)
    gr.Markdown("### Portfolio summary")
    out_summary = gr.Dataframe(wrap=True)

    out_status = gr.Textbox(label="Status", interactive=False)

    def _set_latest():
        return str(DATA_MAX) if DATA_MAX is not None else ""
    btn_latest.click(_set_latest, outputs=asof_txt)

    def _set_2y():
        if DATA_MAX is None:
            return ""
        d = pd.to_datetime(DATA_MAX) - pd.DateOffset(years=2)
        return clamp_date(d).date().isoformat()
    btn_2y.click(_set_2y, outputs=asof_txt)

    def _set_2m():
        if DATA_MAX is None:
            return ""
        d = pd.to_datetime(DATA_MAX) - pd.DateOffset(months=2)
        return clamp_date(d).date().isoformat()
    btn_2m.click(_set_2m, outputs=asof_txt)

    def _on_run(asof_str, cap, thr, corr, mth, maxw, tvol, rcw, covw, minp_):
        try:
            if asof_str and str(asof_str).strip():
                asof = pd.to_datetime(str(asof_str)).normalize()
            else:
                asof = pd.to_datetime(DATA_MAX).normalize() if DATA_MAX is not None else pd.to_datetime("today").normalize()
        except Exception:
            return (
                pd.DataFrame(), pd.DataFrame(), pd.DataFrame(),
                None, None,
                f"Invalid date: {asof_str}. Use YYYY-MM-DD."
            )

        if DATA_MIN is not None and DATA_MAX is not None:
            asof = clamp_date(asof)

        try:
            cand, alloc, summ = portfolio_as_of(
                as_of_date=asof,
                capital_usd=float(cap),
                buy_thr_pct=float(thr),
                corr_min=float(corr),
                method=str(mth),
                max_single_w_pct=float(maxw),
                target_ann_vol_pct=float(tvol),
                roll_corr_win=int(rcw),
                roll_cov_win=int(covw),
                min_periods=int(minp_),
            )
            csv_path, md_path = save_slide(asof, alloc, summ)
            status = (
                f"Plan built for {asof.date()} — files saved:\n"
                f"- {csv_path}\n- {md_path}"
            )
            return cand, alloc, summ, csv_path, md_path, status
        except Exception as e:
            return (
                pd.DataFrame(), pd.DataFrame(), pd.DataFrame(),
                None, None,
                f"Error: {e}"
            )

    run_btn.click(
        _on_run,
        inputs=[asof_txt, capital, pos_thr, corr_gate, method, max_w, targ_vol, roll_corr, roll_cov, minp],
        outputs=[out_candidates, out_alloc, out_summary, out_csv, out_md, out_status],
    )

if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
